In [2]:
URI      = "neo4j+s://a2ebcc41.databases.neo4j.io"  # your AuraDB URI
USER     = "a2ebcc41"
PASSWORD = "T3s9MuSCng4DbwfrMtpfzJwqCtE1tO5EGaqoQOkkJdo" 

import sqlite3
import pandas as pd
from neo4j import GraphDatabase


driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

# ── Load from SQLite ──────────────────────────────────────────────────────────
conn     = sqlite3.connect("delays.db")
schedule = pd.read_sql("SELECT * FROM schedule", conn)
trains   = pd.read_sql("SELECT * FROM trains", conn)
stations = pd.read_sql("SELECT * FROM stations", conn)
conn.close()

print(f"Loaded: {len(trains)} trains, {len(stations)} stations, {len(schedule)} stops")

# ── Step 1: Create Station nodes ──────────────────────────────────────────────
def create_stations(tx, batch):
    tx.run("""
        UNWIND $rows AS row
        MERGE (s:Station {name: row.station_name})
        SET s.train_count   = row.train_count,
            s.junction_rank = row.junction_rank
    """, rows=batch)

station_rows = stations[["station_name","train_count","junction_rank"]].to_dict("records")

with driver.session() as session:
    BATCH = 500
    for i in range(0, len(station_rows), BATCH):
        session.execute_write(create_stations, station_rows[i:i+BATCH])
    print(f"Created {len(station_rows)} Station nodes")

# ── Step 2: Create Train nodes ────────────────────────────────────────────────
def create_trains(tx, batch):
    tx.run("""
        UNWIND $rows AS row
        MERGE (t:Train {number: toString(row.train_no)})
        SET t.name         = row.train_name,
            t.origin       = row.source_station,
            t.destination  = row.destination_station,
            t.distance_km  = row.distance,
            t.running_days = row.days_of_week,
            t.classes      = row.classes
    """, rows=batch)

train_rows = trains.to_dict("records")

with driver.session() as session:
    BATCH = 500
    for i in range(0, len(train_rows), BATCH):
        session.execute_write(create_trains, train_rows[i:i+BATCH])
    print(f"Created {len(train_rows)} Train nodes")

# ── Step 3: Create STOPS_AT relationships ─────────────────────────────────────
# Each stop = (Train)-[:STOPS_AT {sequence, arrival, departure}]->(Station)
def create_stops(tx, batch):
    tx.run("""
        UNWIND $rows AS row
        MATCH (t:Train   {number: toString(row.train_no)})
        MATCH (s:Station {name: row.station_name})
        MERGE (t)-[r:STOPS_AT {sequence: row.stop_sequence}]->(s)
        SET r.arrival_time   = row.arrival_time,
            r.departure_time = row.departure_time,
            r.is_origin      = row.is_origin,
            r.is_destination = row.is_destination
    """, rows=batch)

stop_rows = schedule.to_dict("records")

with driver.session() as session:
    BATCH = 1000
    total = len(stop_rows)
    for i in range(0, total, BATCH):
        session.execute_write(create_stops, stop_rows[i:i+BATCH])
        print(f"  stops: {min(i+BATCH, total):,}/{total:,}", end="\r")
    print(f"\nCreated {total:,} STOPS_AT relationships")

# ── Step 4: Create indexes for fast queries ───────────────────────────────────
with driver.session() as session:
    session.run("CREATE INDEX station_name IF NOT EXISTS FOR (s:Station) ON (s.name)")
    session.run("CREATE INDEX train_number IF NOT EXISTS FOR (t:Train)   ON (t.number)")
    print("Indexes created")

# ── Step 5: Verify with a sample query ───────────────────────────────────────
print("\nVerification queries:")

with driver.session() as session:

    # How many nodes and relationships?
    r = session.run("MATCH (s:Station) RETURN count(s) AS n").single()
    print(f"  Station nodes:       {r['n']:,}")

    r = session.run("MATCH (t:Train) RETURN count(t) AS n").single()
    print(f"  Train nodes:         {r['n']:,}")

    r = session.run("MATCH ()-[r:STOPS_AT]->() RETURN count(r) AS n").single()
    print(f"  STOPS_AT edges:      {r['n']:,}")

    # Top 5 junctions
    print("\n  Top 5 junctions (by trains passing through):")
    result = session.run("""
        MATCH (t:Train)-[:STOPS_AT]->(s:Station)
        RETURN s.name AS station, count(t) AS trains
        ORDER BY trains DESC LIMIT 5
    """)
    for rec in result:
        print(f"    {rec['station']:<30} {rec['trains']} trains")

    # Full route of one train
    print("\n  Route of MANDOVI EXPRESS (10103):")
    result = session.run("""
        MATCH (t:Train {number: '10103'})-[r:STOPS_AT]->(s:Station)
        RETURN s.name AS station, r.sequence AS seq,
               r.arrival_time AS arr, r.departure_time AS dep
        ORDER BY r.sequence
    """)
    for rec in result:
        arr = rec['arr'] or "start"
        dep = rec['dep'] or "end"
        print(f"    {rec['seq']:>2}. {rec['station']:<25} arr={arr:<6}  dep={dep}")

driver.close()
print("\nNeo4j load complete.")# your AuraDB password

Loaded: 8366 trains, 9394 stations, 177431 stops
Created 9394 Station nodes
Created 8366 Train nodes
  stops: 177,431/177,431
Created 177,431 STOPS_AT relationships
Indexes created

Verification queries:
  Station nodes:       9,394
  Train nodes:         8,366
  STOPS_AT edges:      177,431

  Top 5 junctions (by trains passing through):
    Itarsi Jn                      382 trains
    Kanpur Central                 327 trains
    Khurda Road Jn                 323 trains
    Dd Upadhyaya Jn                321 trains
    Ghaziabad                      317 trains

  Route of MANDOVI EXPRESS (10103):
     0. C Shivaji Maharaj T       arr=start   dep=07:10
     1. Dadar                     arr=07:22   dep=07:25
     2. Thane                     arr=07:51   dep=07:55
     3. Panvel                    arr=08:25   dep=08:30
     4. Mangaon                   arr=10:14   dep=10:16
     5. Khed                      arr=11:18   dep=11:20
     6. Chiplun                   arr=11:46   dep=11:48


In [1]:
import sqlite3
import pandas as pd
from neo4j import GraphDatabase
from collections import defaultdict
import json

URI      = "neo4j+s://a2ebcc41.databases.neo4j.io"  # your AuraDB URI
USER     = "a2ebcc41"
PASSWORD = "T3s9MuSCng4DbwfrMtpfzJwqCtE1tO5EGaqoQOkkJdo" 


driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
conn   = sqlite3.connect("delays.db")

# ─────────────────────────────────────────────────────────────────────────────
# ANALYSIS 1: Junction Vulnerability Score
# For each junction, how many unique train-pairs share it?
# A train-pair = two different trains that both stop at the same station.
# If Train A is delayed at that station, Train B (arriving soon after) is at risk.
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("ANALYSIS 1: Junction Vulnerability Score")
print("=" * 60)

query1 = """
MATCH (t1:Train)-[r1:STOPS_AT]->(s:Station)<-[r2:STOPS_AT]-(t2:Train)
WHERE t1.number < t2.number
WITH s, count(*) AS shared_pairs
ORDER BY shared_pairs DESC
LIMIT 30
RETURN s.name AS station, s.train_count AS total_trains, shared_pairs
"""

with driver.session() as session:
    result = session.run(query1)
    vuln = pd.DataFrame([dict(r) for r in result])

print(vuln.to_string(index=False))
vuln.to_sql("junction_vulnerability", conn, if_exists="replace", index=False)

# ─────────────────────────────────────────────────────────────────────────────
# ANALYSIS 2: Cascade Depth — how far does a junction's influence spread?
# For top 10 junctions: how many stations are reachable within 3 hops?
# i.e. trains that pass through Itarsi also pass through which other stations?
# A delay at Itarsi touches ALL those downstream stations.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("ANALYSIS 2: Cascade Depth (3-hop reach)")
print("=" * 60)

top_junctions = vuln["station"].head(10).tolist()

query2 = """
MATCH (s:Station {name: $station})<-[:STOPS_AT]-(t:Train)-[:STOPS_AT]->(s2:Station)
WHERE s2.name <> $station
RETURN count(DISTINCT s2) AS reachable_stations,
       count(DISTINCT t)  AS carrier_trains
"""

depth_rows = []
with driver.session() as session:
    for jn in top_junctions:
        r = session.run(query2, station=jn).single()
        depth_rows.append({
            "junction":           jn,
            "reachable_stations": r["reachable_stations"],
            "carrier_trains":     r["carrier_trains"],
        })

depth_df = pd.DataFrame(depth_rows)
print(depth_df.to_string(index=False))
depth_df.to_sql("cascade_depth", conn, if_exists="replace", index=False)

# ─────────────────────────────────────────────────────────────────────────────
# ANALYSIS 3: Shared-Junction Train Pairs with time proximity
# For each junction, find train pairs where one arrives within 60 min of another.
# These are the HIGH-RISK pairs — a delay in Train A directly threatens Train B.
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("ANALYSIS 3: High-Risk Train Pairs at Top Junctions")
print("=" * 60)

query3 = """
MATCH (t1:Train)-[r1:STOPS_AT]->(s:Station {name: $station})<-[r2:STOPS_AT]-(t2:Train)
WHERE t1.number < t2.number
  AND r1.arrival_time IS NOT NULL
  AND r2.arrival_time IS NOT NULL
RETURN t1.number AS train1, t1.name AS name1, r1.arrival_time AS arr1,
       t2.number AS train2, t2.name AS name2, r2.arrival_time AS arr2,
       s.name AS junction
LIMIT 200
"""

def time_to_minutes(t):
    try:
        h, m = t.split(":")
        return int(h) * 60 + int(m)
    except:
        return None

pair_rows = []
with driver.session() as session:
    for jn in top_junctions[:5]:   # top 5 junctions
        result = session.run(query3, station=jn)
        for r in result:
            m1 = time_to_minutes(r["arr1"])
            m2 = time_to_minutes(r["arr2"])
            if m1 is None or m2 is None:
                continue
            gap = abs(m1 - m2)
            if gap <= 60:   # within 60 minutes of each other
                pair_rows.append({
                    "junction":   r["junction"],
                    "train1":     r["train1"],
                    "name1":      r["name1"],
                    "arr1":       r["arr1"],
                    "train2":     r["train2"],
                    "name2":      r["name2"],
                    "arr2":       r["arr2"],
                    "gap_min":    gap,
                    "risk":       "HIGH" if gap <= 20 else "MEDIUM",
                })

pairs_df = pd.DataFrame(pair_rows).sort_values("gap_min")
print(f"Found {len(pairs_df)} high-risk train pairs across top 5 junctions")
print(f"  HIGH risk (<=20 min gap):   {len(pairs_df[pairs_df['risk']=='HIGH'])}")
print(f"  MEDIUM risk (21-60 min gap): {len(pairs_df[pairs_df['risk']=='MEDIUM'])}")
print("\nTop 15 closest pairs:")
print(pairs_df.head(15)[["junction","name1","arr1","name2","arr2","gap_min","risk"]].to_string(index=False))
pairs_df.to_sql("high_risk_pairs", conn, if_exists="replace", index=False)

# ─────────────────────────────────────────────────────────────────────────────
# ANALYSIS 4: Contagion Index
# The single most important metric of the project.
# For each junction: (shared_pairs × reachable_stations) / total_trains
# Higher = more dangerous when delayed
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("ANALYSIS 4: Contagion Index (the key finding)")
print("=" * 60)

merged = vuln.merge(depth_df, left_on="station", right_on="junction", how="inner")
merged["contagion_index"] = (
    (merged["shared_pairs"] * merged["reachable_stations"])
    / merged["total_trains"]
).round(2)
merged = merged.sort_values("contagion_index", ascending=False)

print(merged[["station","total_trains","shared_pairs","reachable_stations","contagion_index"]]
      .head(15).to_string(index=False))

merged.to_sql("contagion_index", conn, if_exists="replace", index=False)

print("\n*** THE FINDING ***")
top = merged.iloc[0]
print(f"Most dangerous junction: {top['station']}")
print(f"  {int(top['total_trains'])} trains pass through it")
print(f"  A delay there creates {int(top['shared_pairs'])} at-risk train pairs")
print(f"  and ripples across {int(top['reachable_stations'])} downstream stations")
print(f"  Contagion Index: {top['contagion_index']}")

conn.close()
driver.close()
print("\nAll results saved to delays.db")
print("Tables: junction_vulnerability, cascade_depth, high_risk_pairs, contagion_index")

ANALYSIS 1: Junction Vulnerability Score
        station  total_trains  shared_pairs
      Itarsi Jn           382         72771
 Kanpur Central           327         53301
 Khurda Road Jn           323         52003
Dd Upadhyaya Jn           321         51360
      Ghaziabad           317         50086
    Vadodara Jn           313         48828
      Kalyan Jn           312         48516
       Anand Jn           311         48205
  Vijayawada Jn           310         47895
 Ambala Cant Jn           288         41328
      New Delhi           285         40470
       Patna Jn           284         40186
    Bhubaneswar           281         39340
   Kharagpur Jn           279         38781
    Bhusaval Jn           273         37128
     Katpadi Jn           268         35778
     Asansol Jn           265         34980
      Howrah Jn           261         33930
     Barddhaman           257         32896
     Rajamundry           249         30876
      Moradabad           246      